# Managed Tables
Starter notebook for working with managed tables in Databricks.

In [ ]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark

spark.sql("SELECT current_catalog(), current_schema()").show(truncate=False)

In [ ]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

path = "/Volumes/population_metrics/landing/datasets/countries_dataset/countries_dataset/csv_data/countries_population/"
schema = StructType([
    StructField("COUNTRY_ID", IntegerType(), True),
    StructField("NAME", StringType(), True),
    StructField("NATIONALITY", StringType(), True),
    StructField("CODE", StringType(), True),
    StructField("ISO_ALPHA3", StringType(), True),
    StructField("CAPITAL", StringType(), True),
    StructField("POPULATION", IntegerType(), True),
    StructField("AREA_KM2", DoubleType(), True),
    StructField("REGION_ID", IntegerType(), True),
    StructField("SUB_REGION_ID", IntegerType(), True)
])

catalog_name = spark.sql("SELECT current_catalog() AS c").collect()[0]["c"]
schema_name = spark.sql("SELECT current_schema() AS s").collect()[0]["s"]
table_name = "country_population"
full_table_name = f"`{catalog_name}`.`{schema_name}`.`{table_name}`"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.`{schema_name}`")

df = spark.read.format("csv").options(header="true").schema(schema).load(path)
(
    df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable(full_table_name)
)

spark.sql(f"SELECT COUNT(*) AS row_count FROM {full_table_name}").show(truncate=False)
spark.table(full_table_name).show(20, truncate=False)
print(f"Saved managed table: {full_table_name}")

In [ ]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark

query = """
SELECT *
FROM dbw_dataengineering_sandbox_euwest_001.default.country_population
LIMIT 50
"""

spark.sql(query).show(truncate=False)